# SheafPatternFusion Phase 2.5 - WP2.5.4 forced cyclic-poset stratum, shard 01/4

Generates instances whose realized pattern poset is CYCLIC (impossible under Phase-2 sampling: interior mechanism probabilities realize every pattern, and the full simplex of observed sets is Berge-acyclic) via exact 0/1 indicator pins - triangle/square templates plus pinned rejection sampling - then runs the identical Phase-2 pipeline on each accepted instance.

Code: pip-installed from hugogobato/sheafpatternfusion @ v0.3.0.
Runtime: CPU-only (~2 cores). Expected wall time: ~1 h.

The first cell installs the pinned package and restarts the kernel once (required after upgrading numpy/scipy in place). After the runtime reconnects, run Runtime > Run all again; the install cell detects the pins and skips.

In [ ]:
import importlib.metadata as md
import os
import subprocess
import sys

WANT = {'numpy': '2.4.3', 'scipy': '1.17.1', 'sheafpatternfusion': '0.3.0'}
TAG = 'v0.3.0'
REPO = 'https://github.com/hugogobato/sheafpatternfusion.git'


def _ver(pkg):
    try:
        return md.version(pkg)
    except Exception:
        return None


if all(_ver(p) == v for p, v in WANT.items()):
    print('environment OK:', WANT)
else:
    print('installing sheafpatternfusion@' + TAG + ' (one-time per session) ...')
    res = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                          f'git+{REPO}@{TAG}'])
    if res.returncode != 0:
        raise RuntimeError('pip install failed; see log above')
    print('installed -> restarting the kernel so the ABI-matched numpy/scipy')
    print('binaries load cleanly.')
    print('When the runtime reconnects, run Runtime > Run all again; this')
    print('cell will detect the pins and skip.')
    os.kill(os.getpid(), 9)


In [ ]:
import functools
import json
import multiprocessing as mp
import os
import pathlib
import time

os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'


In [ ]:
CYCLIC_CFG = json.loads(r'''{"description": "WP2.5.4 forced cyclic-poset stratum grid. Frozen before any Phase-2.5 run. Engine/fiber budgets identical to configs/phase2/grid.json; only the stratum construction is new.", "shards": 4, "target_per_shard": 140, "honest_attempt_floor_total": 200, "max_attempts": 40000, "seed_base": 20350901, "proposal_mix": {"triangle_n3": 0.45, "square_n4": 0.25, "rejection_n3": 0.2, "rejection_n4": 0.1}, "budgets": {"witness_starts": 24, "jump_starts": 40, "fiber_starts": 48, "max_roots": 12, "undecided_round2_multiplier": 2, "lp_pinch_tol": 1e-09, "lp_width_tol": 0.001, "spread_tol": 1e-06}, "ci_discovery_draws": 16, "seeds": {"draw_seed_base": 20360901}, "pre_registered_gates": {"within_stratum_agreement_min": 0.98, "obstruction_signature_bounds": "(0.0, 1.0) exclusive", "kill_rule_1": "unexplained mismatches > 2% after one debug round -> C1/C2 dead as claimed", "kill_rule_2": "uniformly degenerate obstruction signature -> strip cohomology vocabulary, retitle exhaustive-small-poset claim"}, "outputs": ["results/phase25/cyclic_instances.jsonl", "results/phase25/cyclic_summary.json"]}''')


In [ ]:
SHARD_IDX = 1
N_SHARDS = 4


In [ ]:

OUT_DIR = pathlib.Path('/content/results/phase25')
OUT_DIR.mkdir(parents=True, exist_ok=True)


def load_done(path, key_fn):
    done = set()
    if path.exists():
        for line in path.read_text().splitlines():
            try:
                rec = json.loads(line)
                done.add(key_fn(rec))
            except Exception:
                pass
    return done


def pooled_map(worker_fn, items, n_workers=2,
               stall_timeout_s=2400):
    '''Yield worker_fn(item) for all items, 2-process pool with a stall
watchdog: if no future completes within stall_timeout_s, workers are killed
and the remainder runs sequentially. Worker_fn must be picklable (an
importable function or a functools.partial thereof).'''
    from concurrent.futures import ProcessPoolExecutor, FIRST_COMPLETED, wait

    items = list(items)
    if len(items) <= 1 or n_workers <= 1:
        for it in items:
            yield worker_fn(it)
        return
    ctx = mp.get_context('spawn')
    ex = ProcessPoolExecutor(max_workers=n_workers, mp_context=ctx)
    done_count = 0
    try:
        futs = {ex.submit(worker_fn, it): it for it in items}
        pending = set(futs)
        while pending:
            done_set, pending = wait(pending, timeout=stall_timeout_s,
                                     return_when=FIRST_COMPLETED)
            if not done_set:
                raise RuntimeError(
                    f'pool stalled {stall_timeout_s}s with '
                    f'{len(pending)} futures pending')
            for f in done_set:
                done_count += 1
                yield f.result()
        ex.shutdown(wait=False, cancel_futures=True)
    except Exception as e:
        print(f'(pool yielded {done_count}/{len(items)} results, then '
              f'{type(e).__name__}; finishing remainder sequentially)',
              flush=True)
        for proc in (getattr(ex, '_processes', None) or {}).values():
            try:
                proc.kill()
            except Exception:
                pass
        ex.shutdown(wait=False, cancel_futures=True)
        for it in items[done_count:]:
            yield worker_fn(it)


In [ ]:

from sheafpatternfusion.cyclic_synth import make_cyclic_jobs
from sheafpatternfusion.workers import run_cyclic_job

jobs_all, stats = make_cyclic_jobs(CYCLIC_CFG, shard_idx=SHARD_IDX)
print('generation stats:', stats)
inst_path = OUT_DIR / f'cyclic_instances_shard{SHARD_IDX:02d}.jsonl'
done = load_done(inst_path, lambda r: r['instance_id'])
pending = [j for j in jobs_all if j['iid'] not in done]
print(f'{len(jobs_all)} accepted jobs; {len(done)} instances on file; {len(pending)} to go')
# === RUN ===

if pending:
    tp0 = time.time()
    pilot_recs = run_cyclic_job(dict(pending[0]), CYCLIC_CFG['budgets'],
                                int(CYCLIC_CFG['ci_discovery_draws']))
    per = time.time() - tp0
    eta_min = per * len(pending) / 2 / 60
    print(f'self-pilot: {per:.1f}s/job -> projected ~{eta_min:.0f} min on 2 workers '
          f'({len(pending)} jobs); continuing', flush=True)
    with open(inst_path, 'a') as fout:
        for r in pilot_recs:
            fout.write(json.dumps(r) + '\n')
    pending = pending[1:]

mismatch_instances = 0
completed = 0
t0 = time.time()
fout = open(inst_path, 'a')
buf = []
worker = functools.partial(run_cyclic_job, budgets=CYCLIC_CFG['budgets'],
                           ci_draws=int(CYCLIC_CFG['ci_discovery_draws']))
for recs in pooled_map(worker, pending, n_workers=2):
    agree = all(((r['gt_recoverable'] == 'RECOVERABLE') ==
                 (r['sheaf_recoverable'] == 'RECOVERABLE')) for r in recs)
    mismatch_instances += int(not agree)
    buf.extend(recs)
    completed += 1
    if len(buf) >= 8:
        for r in buf:
            fout.write(json.dumps(r) + '\n')
        buf = []
        fout.flush()
    if completed % 10 == 0:
        el = time.time() - t0
        print(f'[{completed}/{len(pending)}] {el/60:.1f} min, '
              f'mismatch_inst={mismatch_instances}', flush=True)
for r in buf:
    fout.write(json.dumps(r) + '\n')
fout.close()
(OUT_DIR / f'cyclic_gen_stats_shard{SHARD_IDX:02d}.json').write_text(
    json.dumps({'generation': stats, 'ran': completed + 1,
                'mismatch_instances': mismatch_instances}, indent=1))
print('CYCLIC SHARD DONE')


In [ ]:
import glob
output_files = sorted(glob.glob(str(OUT_DIR / '*.jsonl')) + glob.glob(str(OUT_DIR / '*.json')))
for output_file in output_files:
    try:
        from google.colab import files
        files.download(output_file)
        print('Downloaded:', output_file)
    except Exception as e:
        print('(Not on Colab / download skipped):', e)
